# Pipeline for scale shape extraction

In [ ]:
!pip list | grep onnxruntime-gpu

#Core code

In [1]:
from pathlib import Path
import skimage.io as io
import matplotlib.pyplot as plt
import numpy as np
from skimage import img_as_ubyte
from skimage.measure import find_contours
from skimage.segmentation import find_boundaries

In [ ]:
# @title Scale mask segmentation
from rembg import remove
from skimage.morphology import binary_erosion, disk
from skimage.measure import label, regionprops
from skimage.morphology import reconstruction, remove_small_holes
from skimage.segmentation import find_boundaries

def extract_scale_mask(img, alpha_threshold=128, alpha_output=False):
  # Apply rembg to remove the background
  output_img_rgba = remove(img)

  # Extract the alpha channel which serves as the mask
  alpha_channel = output_img_rgba[:, :, 3]

  # Create a binary mask (0 or 255) from the alpha channel
  binary_mask = (alpha_channel >= alpha_threshold).astype(bool)

  # Label connected components in the binary mask
  labeled_mask = label(binary_mask)

  # Get region properties
  regions = regionprops(labeled_mask)

  # Find the largest region by area
  largest_region = None
  if regions:
      largest_region = max(regions, key=lambda r: r.area)

  # Create a new mask containing only the largest region directly from its label
  largest_region_mask = np.zeros_like(binary_mask, dtype=np.uint8)
  if largest_region is not None:
      largest_region_mask = (labeled_mask == largest_region.label)
      largest_region_mask = remove_small_holes( largest_region_mask, max_size = 10000 )


  if (alpha_output):
    # Do morphological reconstruction to get alpha mask only for the largest region
    alpha_uint8 = img_as_ubyte(alpha_channel)
    mask_uint8 = img_as_ubyte(largest_region_mask)
    seed_uint8 = np.minimum( alpha_uint8, mask_uint8, dtype=np.uint8 )  # Seed cannot be larger than reconstruction mask (alpha)
    largest_region_alpha = reconstruction(seed_uint8, alpha_uint8, method='dilation').astype(np.uint8)

    return largest_region_mask, largest_region_alpha
  else:
    return largest_region_mask

def overlay_contour(img, mask, color=(255,0,0)):

  # Find the boundaries of the mask
  contour_pixels = find_boundaries(mask.astype(bool), mode='outer')

  # Create a copy of the image to draw the contour on
  img_with_contour = img.copy()

  # Set the contour pixels to red
  # img_with_contour expects RGB, so [R, G, B] = [255, 0, 0] for red
  img_with_contour[contour_pixels] = color

  return img_with_contour

In [3]:
# @title Contour extraction functions
from skimage import measure

def extract_contour(mask):
    contours = measure.find_contours(mask, level=0.5)
    if not contours:
        raise ValueError("No contour found")

    contour = max(contours, key=len)

    # convert (row,col) → (x,y)
    contour = contour[:, ::-1]

    # ensure closed
    if not np.allclose(contour[0], contour[-1]):
        contour = np.vstack([contour, contour[0]])

    return contour

from scipy.interpolate import interp1d

def resample_contour(contour, n_points=300):
    deltas = np.diff(contour, axis=0)
    dists = np.sqrt((deltas**2).sum(axis=1))
    cumulative = np.concatenate([[0], np.cumsum(dists)])

    total_length = cumulative[-1]
    cumulative /= total_length

    fx = interp1d(cumulative, contour[:,0])
    fy = interp1d(cumulative, contour[:,1])

    uniform = np.linspace(0, 1, n_points)
    resampled = np.column_stack([fx(uniform), fy(uniform)])

    return resampled

def ensure_clockwise(contour):
    x = contour[:,0]
    y = contour[:,1]
    area = np.sum(x[:-1]*y[1:] - x[1:]*y[:-1])
    if area > 0:
        contour = contour[::-1]
    return contour

from scipy.interpolate import splprep

def fit_periodic_spline(contour, smooth=1.0):
    x = contour[:,0]
    y = contour[:,1]

    tck, u = splprep([x, y], s=smooth, per=True)
    return tck

def full_outline_pipeline(mask,
                          n_points=300,
                          smooth=1.0):
    contour = extract_contour(mask)
    contour = resample_contour(contour, n_points)
    contour = ensure_clockwise(contour)
    spline = fit_periodic_spline(contour, smooth)

    return contour, spline

In [4]:
# @title Contour curvature normalization
from scipy.interpolate import splprep, splev

def spline_curvature(contour, smooth=0):
    tck, u = splprep([contour[:,0], contour[:,1]], s=smooth, per=True)

    dx, dy = splev(u, tck, der=1)
    ddx, ddy = splev(u, tck, der=2)

    numerator = np.array(dx)*np.array(ddy) - np.array(dy)*np.array(ddx)
    denominator = (np.array(dx)**2 + np.array(dy)**2)**1.5

    curvature = numerator / denominator
    return curvature

import numpy as np

import numpy as np

def rotate_contour_by_max_curvature(contour, curvature, kind='absmax'):
    """
    Rotate a closed contour so that the point with the highest |curvature| becomes the first point.
    Curvature is computed internally using finite differences.

    Parameters
    ----------
    contour : (N,2) array
        x,y coordinates of the contour (may be closed)

    Returns
    -------
    rotated_contour : (N,2) array
        rotated, closed contour
    curvature : (N,) array
        curvature along rotated contour
    peak_idx : int
        index of max |curvature| in original contour
    """

    # Remove duplicate endpoint if contour is closed
    if np.allclose(contour[0], contour[-1]):
        contour = contour[:-1]

    # Find max absolute curvature
    if (kind=='min'):
      peak_idx = np.argmin(curvature)
    elif (kind=='max'):
      peak_idx = np.argmax(curvature)
    elif (kind=='absmax'):
      peak_idx = np.argmax(np.abs(curvature))
    else:
      raise 'invalid kind'

    # Circularly rotate contour and curvature
    rotated_contour = np.roll(contour, -peak_idx, axis=0)
    rotated_curvature = np.roll(curvature, -peak_idx)

    # Re-close contour
    rotated_contour = np.vstack([rotated_contour, rotated_contour[0]])

    return rotated_contour, rotated_curvature, peak_idx

In [5]:
# @title Code for contour split into subarcs
import numpy as np

def main_axes_from_anchor(contour, anchor):
    """
    Compute main axes of a shape using a custom anchor point.

    Parameters
    ----------
    contour : (N,2) array
        x,y coordinates of the shape (closed or open)
    anchor : (2,) array-like
        The anchor/origin point (x0, y0)

    Returns
    -------
    eigvals : array, shape (2,)
        Eigenvalues of covariance matrix (variance along each axis)
    eigvecs : array, shape (2,2)
        Eigenvectors (columns) corresponding to main axes
    """
    # Shift coordinates
    X = contour[:,0] - anchor[0]
    Y = contour[:,1] - anchor[1]

    coords = np.column_stack([X, Y])

    # Covariance
    C = np.cov(coords.T)

    # Eigen decomposition
    eigvals, eigvecs = np.linalg.eigh(C)  # ascending order
    # largest eigenvalue first
    idx = eigvals.argsort()[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    displacement = np.mean(coords, axis=0)
    if np.dot(displacement, eigvecs[:,0]) < 0:
        eigvecs[:,0] *= -1  # flip direction
    # Second eigenvector orthogonal
    eigvecs[:,1] = np.array([-eigvecs[1,0], eigvecs[0,0]])

    return eigvals, eigvecs

def rotate_coords_main_axes(contour, eigvecs):
  # Shift coordinates
  XY = contour - anchor.reshape(1,2)

  RXY = XY @ eigvecs

  return RXY

import numpy as np

def extreme_points_perpendicular(contour, anchor, eigvecs):
    """
    Find the two points furthest from the main axis (first eigenvector) on each side.

    Parameters
    ----------
    contour : (N,2) array
    anchor : (2,) array
    eigvecs : (2,2) array
        Columns = eigenvectors, first = main axis

    Returns
    -------
    pt_pos_side : (2,) array
        Point on positive side of axis furthest away
    pt_neg_side : (2,) array
        Point on negative side of axis furthest away
    distances : (N,) array
        perpendicular distances of all points
    """
    v0 = eigvecs[:,0]  # main axis
    displacement = contour - anchor  # (N,2)

    # Signed perpendicular distance in 2D
    # Using cross product to get sign: v0 x disp
    # cross2D(a,b) = a_x*b_y - a_y*b_x
    signed_dist = v0[0]*displacement[:,1] - v0[1]*displacement[:,0]

    # Perpendicular distance magnitude
    #distances = np.abs(signed_dist)

    # Split by side
    #pos_mask = signed_dist > 0
    #neg_mask = signed_dist < 0

    id_right = np.argmax(signed_dist)
    id_left  = np.argmin(signed_dist)

    #pt_pos_side = contour[pos_mask][np.argmax(distances[pos_mask])]
    #pt_neg_side = contour[neg_mask][np.argmax(distances[neg_mask])]

    return id_right, id_left

import numpy as np

def low_curvature_intervals_include_points(curvature, threshold, indices_to_include):
    """
    Find contiguous intervals where curvature < threshold that include given indices.
    Works for a closed contour.

    Parameters
    ----------
    curvature : (N,) array
        curvature vector
    threshold : float
        threshold for "low curvature"
    indices_to_include : list or array of int
        indices that must be included in the intervals

    Returns
    -------
    intervals : list of tuples
        Each tuple = (start_idx, end_idx) of low-curvature interval (inclusive start, exclusive end)
    """
    N = len(curvature)

    # 1️⃣ Boolean mask of low curvature
    low_mask = curvature < threshold

    # 2️⃣ Duplicate mask to handle circular wrap-around
    doubled = np.concatenate([low_mask, low_mask])

    intervals = []
    start = None
    for i, val in enumerate(doubled):
        if val and start is None:
            start = i
        elif not val and start is not None:
            intervals.append((start, i))
            start = None
    if start is not None:
        intervals.append((start, len(doubled)))

    # 3️⃣ Map intervals back to original index space
    valid_intervals = []
    for s, e in intervals:
        length = e - s
        if length > 0:
            s_mod = s % N
            valid_intervals.append((s_mod, length))

    # 4️⃣ Keep only intervals that contain the specified indices
    result = []
    for idx in indices_to_include:
        # find interval containing idx
        found = False
        for s, length in valid_intervals:
            interval_indices = np.arange(s, s+length) % N
            if idx in interval_indices:
                result.append((s, (s+length)%N))
                found = True
                break
        if not found:
            result.append(None)  # fallback if no interval includes this index

    return result

def local_extreme_x_intervals(up_contour, seeds):
    N = len(curvature)

    y = np.concatenate([up_contour[:,0], up_contour[:,0]])  # double for easy boundary handling

    intervals = []
    for seed in seeds:
        # ---- Forward search ----
        idx_forward = seed
        for i in range(seed + 1, N - 1):
            if not (y[i-1] <= y[i]) ^ (y[i] >= y[i+1]): # any weak extremum
                idx_forward = i
                break

        # ---- Backward search ----
        idx_backward = seed
        for i in range(seed - 1, 0, -1):
            if not (y[i-1] <= y[i]) ^ (y[i] >= y[i+1]): # any weak extremum
                idx_backward = i
                break

        intervals.append( (idx_backward, idx_forward) )

    return intervals

def plot_serration_cut(up_contour, id_right_max, id_left_max, id_right_expanded, id_left_expanded):
  p1 = up_contour[id_right_max,:]
  p2 = up_contour[id_left_max,:]
  p1e = up_contour[id_right_expanded,:]
  p2e = up_contour[id_left_expanded,:]

  plt.plot( [p1[0],p2[0]], [p1[1],p2[1]], 'b*-' )
  plt.plot( [p1e[0],p2e[0]], [p1e[1],p2e[1]], 'k*-' )

def expand_serration(up_contour, id_right_max, id_left_max):
  p1 = up_contour[id_right_max,:]
  p2 = up_contour[id_left_max,:]
  invslope = (p2[0] - p1[0]) / (p2[1] - p1[1])

  deltas = up_contour[:,0] - invslope * (up_contour[:,1] - p1[1])

  mindelta = np.min( deltas[id_right_max:id_left_max+1] )

  N = len(up_contour)

  # ---- Forward search from left ----
  seed = id_left_max
  idx_forward = seed
  for i in range(seed + 1, N - 1):
      if deltas[i] < mindelta: # further back than threshold
          idx_forward = i
          break

  # ---- Backward search ----
  seed = id_right_max
  idx_backward = seed
  for i in range(seed - 1, 0, -1):
      if deltas[i] < mindelta: # further back than threshold
          idx_backward = i
          break

  return idx_backward, idx_forward  # expand_right, expand_left

import numpy as np

def trim_low_curvature_intervals(curvature, intervals, negative_threshold):
    """
    Trim multiple low-curvature intervals from both ends until curvature
    drops below negative_threshold.

    Parameters
    ----------
    curvature : (N,) array
        curvature vector
    intervals : list of tuples
        Each tuple = (start_idx, end_idx) of a low-curvature interval
        (indices modulo N for closed contour)
    negative_threshold : float
        curvature threshold to stop trimming

    Returns
    -------
    trimmed_intervals : list of tuples
        Each tuple = (new_start_idx, new_end_idx) after trimming
    """
    N = len(curvature)
    trimmed_intervals = []

    for start, end in intervals:
        # create array of indices for the interval
        indices = np.arange(start, end) % N
        curv = curvature[indices]

        # trim from start
        for i, val in enumerate(curv):
            if val > negative_threshold:
                break
        new_start = indices[i]

        # trim from end
        for j, val in enumerate(curv[::-1]):
            if val > negative_threshold:
                break
        new_end = indices[-(j+1)] + 1  # exclusive

        # modulo N for closed contour
        new_start %= N
        new_end %= N

        trimmed_intervals.append((new_start, new_end))

    return trimmed_intervals

import numpy as np

def generate_labels_closed_contour(N, anchor_idx, right_interval, left_interval):
    """
    Generate a label vector for a closed contour with implicit top interval.

    Labels:
        0 = anchor/start
        1 = right interval
        2 = top interval (between right and left)
        3 = left interval

    Parameters
    ----------
    N : int
        Number of points in the contour
    anchor_idx : int
        Index of the starting point
    left_interval : tuple
        (start_idx, end_idx) of left interval
    right_interval : tuple
        (start_idx, end_idx) of right interval

    Returns
    -------
    labels : (N,) array of int
    """
    labels = np.zeros(N, dtype=int)  # default = 0 (anchor)

    def mark_interval(interval, label_id):
        s, e = interval
        idxs = np.arange(s, e+1) % N
        labels[idxs] = label_id

    # assign right and left intervals
    mark_interval(right_interval, 1)
    mark_interval(left_interval, 3)

    top_interval = (right_interval[1]+1,left_interval[0]-1)
    mark_interval(top_interval, 2)

    return labels

from scipy.signal import hilbert

def reconstruct_envelope(signal):
    """
    Given a signal that is a single quadrature component, e.g., cos(phi(t)) * A(t),
    reconstruct the amplitude envelope using the analytic signal.

    Parameters
    ----------
    signal : (N,) array
        Observed cos component of a (cos,sin) quadrature signal

    Returns
    -------
    envelope : (N,) array
        Smoothed amplitude envelope
    """
    # Compute analytic signal
    analytic = hilbert(signal)

    # Amplitude envelope
    envelope = np.abs(analytic)

    return envelope

def split_contour(contour, curvature_threshold=0.02):
  anchor = contour[0]
  eigvals, eigvecs = main_axes_from_anchor(contour, anchor)

  id_right, id_left = extreme_points_perpendicular(contour, anchor, eigvecs)

  envelope = reconstruct_envelope(curvature)

  intervals = low_curvature_intervals_include_points(envelope, threshold=curvature_threshold, indices_to_include=[id_right, id_left])

  trimmed_intervals = intervals
  #trimmed_intervals = trim_low_curvature_intervals(curvature, intervals, negative_threshold=-0.005)

  labels = generate_labels_closed_contour(len(contour), 0, trimmed_intervals[0], trimmed_intervals[1])

  return trimmed_intervals, labels

def plot_split_contours(contour, labels):
  plt.scatter(contour[:,0], contour[:,1], c=labels)
  plt.plot(contour[0,0], contour[0,1], 'r*')  # stem anchor


def smooth_cyclic(curvature, window_size=5):
    """
    Smooth the curvature using a cyclic (circular) convolution.

    Parameters
    ----------
    curvature : (N,) array
        curvature vector
    window_size : int
        Size of smoothing kernel (odd preferred)

    Returns
    -------
    smoothed : (N,) array
        Smoothed |curvature| vector
    """
    #abs_curv = np.abs(curvature)
    N = len(curvature)

    # simple uniform kernel
    kernel = np.ones(window_size) / window_size

    # pad for circular convolution
    pad = window_size // 2
    extended = np.concatenate([curvature[-pad:], curvature, curvature[:pad]])

    # convolve
    smoothed = np.convolve(extended, kernel, mode='valid')

    return smoothed

def plot_up_xy(up_contour, curvature, ids=None, cols=None):

    fig, axes = plt.subplots(
        3, 1,
        sharex=True,
        figsize=(8, 6)
    )

    u = range(len(contour))
    if (ids is not None and cols is None):
      nids = len(ids)
      cols = ['r'] * nids

    # x(u)
    axes[0].plot(u, up_contour[:,0])
    axes[0].set_ylabel("x(u)")
    axes[0].grid(True)
    if (ids is not None):
        for id,col in zip(ids,cols):
          axes[0].plot(id, up_contour[id,0], '*', color=col)

    # y(u)
    axes[1].plot(u, up_contour[:,1])
    axes[1].set_ylabel("y(u)")
    axes[1].grid(True)
    if (ids is not None):
        for id,col in zip(ids,cols):
          axes[1].plot(id, up_contour[id,1], '*', color=col)

    # curvature(u)
    axes[2].plot(u, curvature)
    axes[2].set_ylabel("curvature")
    axes[2].set_xlabel("u (arc-length parameter)")
    axes[2].grid(True)
    if (ids is not None):
        for id,col in zip(ids,cols):
          axes[2].plot(id, curvature[id], '*', color=col)

    plt.tight_layout()

def plot_contour_xy(up_contour, ids=None, cols=None):

    axes = plt.gca()

    u = range(len(contour))
    if (ids is not None and cols is None):
      nids = len(ids)
      cols = ['r'] * nids

    # x(u)
    axes.plot(up_contour[:,0], up_contour[:,1],'.-')
    axes.set_xlabel("x(u)")
    axes.set_ylabel("y(u)")
    axes.grid(True)
    if (ids is not None):
        for id,col in zip(ids,cols):
          axes.plot(up_contour[id,0], up_contour[id,1], '*', color=col)

    plt.tight_layout()


from skimage.draw import circle_perimeter, line, polygon

import numpy as np
import matplotlib.colors as mcolors

def to_rgb255(color):
    """
    Convert various color formats to RGB [0,255].

    Supports:
        - short names: 'r'
        - full names: 'red'
        - hex: '#ff0000'
        - 0-1 floats: (1,0,0)
        - 0-255 ints: (255,0,0)
        - numpy arrays / lists

    Returns
    -------
    np.ndarray shape (3,) dtype uint8
    """
    # If already array-like
    if isinstance(color, (list, tuple, np.ndarray)):
        arr = np.array(color)
        return arr.astype(np.uint8)

    # Otherwise assume string-like → use matplotlib
    rgb01 = mcolors.to_rgb(color)  # returns floats in [0,1]
    return (np.array(rgb01) * 255).astype(np.uint8)

def draw_keypoints( img, contour, ids=None, cols=None ):
    u = range(len(contour))

    if (ids is not None and cols is None):
      nids = len(ids)
      cols = ['r'] * nids

    if (ids is not None):
      for id,col in zip(ids,cols):
          X,Y = np.round(contour[id,0]).astype(int), np.round(contour[id,1]).astype(int)
          rr, cc = circle_perimeter(Y,X, 5, shape=img.shape)
          img[rr, cc] = to_rgb255(col)
          rr, cc = circle_perimeter(Y,X, 6, shape=img.shape)
          img[rr, cc] = to_rgb255(col)

def draw_axis( img, anchor, eigvecs, sz=200, col='b'):
  U = eigvecs[:,0]
  V = eigvecs[:,1]

  col = to_rgb255(col)

  c0,c1,r0,r1 = (anchor[0], anchor[0]+sz*U[0], anchor[1], anchor[1]+sz*U[1])
  #c0,c1,r0,r1 = anchor[0], anchor[0]+10, anchor[1], anchor[1]+10
  c0 = int(c0)
  c1 = int(c1)
  r0 = int(r0)
  r1 = int(r1)
  rr, cc = line( r0,c0,r1,c1 )
  mask = (
      (rr >= 0) & (rr < img.shape[0]) &
      (cc >= 0) & (cc < img.shape[1])
  )
  rr = rr[mask]
  cc = cc[mask]
  img[rr, cc] = col
  #print( rr, cc)


## Global parameters

In [6]:
from pathlib import Path

IMGDIR = Path('/mnt/data/users/jsoto/sample_img')   # change this to your folder
filenames = sorted([p.name for p in IMGDIR.glob('*.jpg')])

print(f"Found {len(filenames)} jpg files")
print(filenames[:5])  # preview first few

Found 16 jpg files
['in6_dorsal_black_area4_cover_scale0001.jpg', 'in6_dorsal_black_area4_cover_scale0001.jpg.overlay.jpg', 'in6_dorsal_black_area4_cover_scale0001.jpg.overlay.jpg.overlay.jpg', 'in6_dorsal_black_area4_cover_scale0001.jpg.overlay.jpg.overlay.jpg.overlay.jpg', 'in6_dorsal_black_area4_cover_scale0002.jpg']


# Process one image manually

In [7]:
id = 0
imgpath = IMGDIR / filenames[id]
maskpath = IMGDIR / ( filenames[id] + '.mask.png' )
overlaypath = IMGDIR / ( filenames[id] + '.overlay.jpg' )

print(imgpath)
print(maskpath)
print(overlaypath)

img = io.imread( imgpath )
#plt.imshow(img)

/mnt/data/users/jsoto/sample_img/in6_dorsal_black_area4_cover_scale0001.jpg
/mnt/data/users/jsoto/sample_img/in6_dorsal_black_area4_cover_scale0001.jpg.mask.png
/mnt/data/users/jsoto/sample_img/in6_dorsal_black_area4_cover_scale0001.jpg.overlay.jpg


In [8]:
# Segment
mask, alpha = extract_scale_mask(img, alpha_threshold=200, alpha_output=True)

overlay = overlay_contour(img, mask, color=(0,0,255))

fig,axes=plt.subplots(1,3, figsize=(16, 4), squeeze=True)
plt.sca(axes[0])
plt.imshow(alpha, cmap='gray')
plt.title('Alpha of scale')
plt.colorbar()

plt.sca(axes[1])
plt.imshow(mask, cmap='gray')
plt.title('Mask of scale')

plt.sca(axes[2])
plt.imshow(img)
plt.contour(mask, levels=[0.5], colors=['blue'])

plt.tight_layout()
plt.show()


# Extract and process contour
contour, spline = full_outline_pipeline(mask,
                          n_points=300,
                          smooth=1.0)

curvature = spline_curvature( contour, smooth=15.0 )

contour, curvature, _ = rotate_contour_by_max_curvature(contour, curvature, kind='min')
u = range(len(contour))

intervals, labels = split_contour(contour, curvature_threshold=0.1)


anchor = contour[0]
eigvals, eigvecs = main_axes_from_anchor(contour, anchor)

up_contour = rotate_coords_main_axes(contour, eigvecs)

id_right, id_left = extreme_points_perpendicular(contour, anchor, eigvecs)
intervals = local_extreme_x_intervals(up_contour, seeds=[id_right, id_left])

id_right_expanded, id_left_expanded = expand_serration(up_contour, intervals[0][1], intervals[1][0])
intervals_expanded = [ [ intervals[0][0], id_right_expanded ], [id_left_expanded, intervals[1][1]] ]

labels = generate_labels_closed_contour(len(contour), 0, intervals_expanded[0], intervals_expanded[1])



plt.figure()
plt.scatter(contour[:,0], contour[:,1], s=1, c=curvature)
plt.gca().get_yaxis().set_inverted(True)
plt.clim(-0.1,0.1)
plt.title('Shape color=curvature')
plt.colorbar()

plt.figure()
plt.scatter(contour[:,0], contour[:,1], s=1, c=range(contour.shape[0]))
plt.gca().get_yaxis().set_inverted(True)
plt.title('Shape color=index')
plt.colorbar()

plt.figure()
plot_split_contours(contour, labels)
plt.plot( contour[id_right,0], contour[id_right,1], 'r*' )
plt.plot( contour[id_left,0], contour[id_left,1], 'g*' )
plt.plot( contour[intervals[0],0], contour[intervals[0],1], 'm*' )
plt.plot( contour[intervals[1],0], contour[intervals[1],1], 'c*' )

plt.figure()
plot_split_contours(up_contour, labels)
#plt.plot(up_contour[:,0], up_contour[:,1], 'b.-' )
plt.gca().yaxis.set_inverted(True)
plot_serration_cut(up_contour, intervals[0][1], intervals[1][0], intervals_expanded[0][1], intervals_expanded[1][0])

u = range(len(contour))
plot_up_xy( up_contour, curvature,
            ids=[0, id_right, id_left, intervals[0][0], intervals[0][1], intervals[1][0], intervals[1][1], intervals_expanded[0][1], intervals_expanded[1][0]],
            cols=['k','r','g','m','m','c','c', 'k','k'] )

plt.figure()
plot_contour_xy( up_contour,
                 ids=[0, id_right, id_left, intervals[0][0], intervals[0][1], intervals[1][0], intervals[1][1], intervals_expanded[0][1], intervals_expanded[1][0]],
                 cols=['k','r','g','m','m','c','c', 'k','k'] )


ids=[0, id_right, id_left, intervals[0][0], intervals[0][1], intervals[1][0], intervals[1][1], intervals_expanded[0][1], intervals_expanded[1][0]]
cols=['k','r','g','m','m','c','c', 'k','k']

sz = np.max(up_contour[:,0]) # max x coordinate, gives the length from the anchor

plt.figure()
#img2 = img.copy()
img2 = overlay_contour(img, mask, color=(0,0,255))
draw_keypoints( img2, contour, ids, cols )
draw_axis( img2, anchor, eigvecs, sz, col='k')
plt.imshow(img2)
print(anchor)
print(eigvecs)

/tmp/ipykernel_511039/1317394320.py:16: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  binary_mask = (alpha_channel >= alpha_threshold).astype(np.bool)


AttributeError: module 'numpy' has no attribute 'bool'.
`np.bool` was a deprecated alias for the builtin `bool`. To avoid this error in existing code, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations

In [ ]:
anchor

# Process list of files

In [ ]:
!pip install h5py

In [ ]:
from skimage import img_as_ubyte
import h5py

contours = []
up_contours = []

for name in filenames:
  imgpath = IMGDIR / name
  alphapath = IMGDIR / ( name + '.alpha.png' )
  maskpath = IMGDIR / ( name + '.mask.png' )
  overlaypath = IMGDIR / ( name + '.overlay.jpg' )
  h5path = IMGDIR / ( name + '.contours.h5' )

  print(f"Processing {imgpath}...")

  img = io.imread(imgpath)

  mask, alpha = extract_scale_mask(img, alpha_threshold=200, alpha_output=True)
  io.imsave(maskpath, img_as_ubyte(mask))
  io.imsave(alphapath, alpha)

  overlay = overlay_contour(img, mask, color=(0,0,255))

  contour, spline = full_outline_pipeline(mask, n_points=300, smooth=1.0)

  curvature = spline_curvature(contour, smooth=15.0)
  contour, curvature, _ = rotate_contour_by_max_curvature(contour, curvature, kind='min')
  u = range(len(contour))

  anchor = contour[0]
  eigvals, eigvecs = main_axes_from_anchor(contour, anchor)

  up_contour = rotate_coords_main_axes(contour, eigvecs)

  id_right, id_left = extreme_points_perpendicular(contour, anchor, eigvecs)
  intervals = local_extreme_x_intervals(up_contour, seeds=[id_right, id_left])

  right_max = intervals[0][1]
  left_max = intervals[1][0]

  print("intervals:", intervals)
  print("right_max:", right_max, "left_max:", left_max)

  try:
      if right_max > left_max:
          raise ValueError(
              f"Invalid interval ordering: right_max={right_max}, left_max={left_max}"
          )

      id_right_expanded, id_left_expanded = expand_serration(
          up_contour, right_max, left_max
      )

  except ValueError as e:
      print(f"Warning: expand_serration failed for {name}: {e}")
      id_right_expanded = right_max
      id_left_expanded = left_max

  intervals_expanded = [
      [intervals[0][0], id_right_expanded],
      [id_left_expanded, intervals[1][1]]
  ]

  labels = generate_labels_closed_contour(len(contour), 0, intervals_expanded[0], intervals_expanded[1])

  sz = np.max(up_contour[:,0])

  ids = [
      0, id_right, id_left,
      intervals[0][0], intervals[0][1],
      intervals[1][0], intervals[1][1],
      intervals_expanded[0][1], intervals_expanded[1][0]
  ]
  cols = ['k','r','g','m','k','k','c','m','c']

  draw_keypoints(overlay, contour, ids, cols)
  draw_axis(overlay, anchor, eigvecs, sz, col='k')

  io.imsave(overlaypath, overlay)

  with h5py.File(h5path, "w") as f:
    f.create_dataset("contour", data=contour)
    f.create_dataset("curvature", data=curvature)
    f.create_dataset("up_contour", data=up_contour)
    f.create_dataset("anchor", data=anchor)
    f.create_dataset("eigvecs", data=eigvecs)
    f.create_dataset("eigvals", data=eigvals)
    f.create_dataset("id_sides", data=[id_right, id_left])
    f.create_dataset("intervals", data=intervals)
    f.create_dataset("intervals_expanded", data=intervals_expanded)

    f.attrs["version"] = "1.0"
    f.attrs["description"] = "Butterfly scale contour analysis"

  contours.append(contour)
  up_contours.append(up_contour)

# Contour analysis

In [ ]:
import h5py

contours = []
up_contours = []

for name in filenames:
  imgpath = IMGDIR / name
  alphapath = IMGDIR / ( name + '.alpha.png' )
  maskpath = IMGDIR / ( name + '.mask.png' )
  overlaypath = IMGDIR / ( name + '.overlay.jpg' )
  h5path = IMGDIR / ( name + '.contours.h5' )

  with h5py.File(h5path, "r") as data:
    up_contour   = data["up_contour"][:]

  up_contours.append(up_contour)

In [ ]:
up_contours

In [ ]:
#!pip install ipywidgets

In [ ]:
plt.figure()
for up_contour in up_contours:
  plt.plot(up_contour[:,0], up_contour[:,1], '.-' )
plt.gca().yaxis.set_inverted(True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# ---- Example contours (replace with yours) ----
# Must have same shape (N,2)
Dcontours = dict(enumerate(up_contours))

# ---- Interpolation function ----
def interpolate_contours(c1, c2, alpha):
    return (1 - alpha) * c1 + alpha * c2

# ---- Plot function ----
def plot_interpolation(id1, id2, alpha):
    contour1 = Dcontours[id1]
    contour2 = Dcontours[id2]

    interp = interpolate_contours(contour1, contour2, alpha)

    plt.figure(figsize=(5,5))
    plt.plot(contour1[:,0], contour1[:,1], '--', label='Contour A')
    plt.plot(contour2[:,0], contour2[:,1], '--', label='Contour B')
    plt.plot(interp[:,0], interp[:,1], linewidth=3, label='Interpolated')
    plt.axis('equal')
    plt.legend()
    plt.title(f'alpha = {alpha:.2f}')
    plt.gca().yaxis.set_inverted(True)
    plt.show()

dropdown1 = widgets.Dropdown(
    options=list(Dcontours.keys()),
    value=list(Dcontours.keys())[0],
    description='Contour A'
)

dropdown2 = widgets.Dropdown(
    options=list(Dcontours.keys()),
    value=list(Dcontours.keys())[1],
    description='Contour B'
)


# ---- Slider ----
alpha_slider = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=1.0,
    step=0.01,
    description='Alpha',
    continuous_update=True
)

# widgets.interact(plot_interpolation, id1=dropdown1,
#                  id2=dropdown2, alpha=alpha_slider);

In [ ]:
!pip install h

In [ ]:
import h5py

with h5py.File(h5path, "w") as f:
    f.create_dataset("contour", data=contour)
    f.create_dataset("curvature", data=curvature)
    f.create_dataset("up_contour", data=up_contour)
    f.create_dataset("anchor", data=anchor)
    f.create_dataset("eigvecs", data=eigvecs)
    f.create_dataset("eigvals", data=eigvals)
    f.create_dataset("id_sides", data=[id_right, id_left])
    f.create_dataset("intervals", data=intervals)
    f.create_dataset("intervals_expanded", data=intervals_expanded)

    # Optional metadata
    f.attrs["version"] = "1.0"
    f.attrs["description"] = "Butterfly scale contour analysis"

# Contour extraction and analysis

In [ ]:
name=filenames[2]
imgpath = IMGDIR / name
alphapath = IMGDIR / ( name + '.alpha.png' )
maskpath = IMGDIR / ( name + '.mask.png' )
overlaypath = IMGDIR / ( name + '.overlay.jpg' )

print(f"Processing {imgpath}...")

img = io.imread( imgpath )

mask, alpha = extract_scale_mask(img, alpha_threshold=200, alpha_output = True)
overlay = overlay_contour(img, mask, color=(0,0,255))

contour, spline = full_outline_pipeline(mask,
                          n_points=300,
                          smooth=1.0)

curvature = spline_curvature( contour, smooth=5.0 )

contour, curvature, _ = rotate_contour_by_max_curvature(contour, curvature, kind='min')
u = range(len(contour))

intervals, labels = split_contour(contour)
plot_split_contours(contour, labels)


In [ ]:
contour, spline = full_outline_pipeline(mask,
                          n_points=300,
                          smooth=1.0)

curvature = spline_curvature( contour, smooth=10.0 )

contour, curvature, _ = rotate_contour_by_max_curvature(contour, curvature, kind='min')
u = range(len(contour))

plt.figure(figsize=(8, 6))
plt.imshow(img)
#plt.plot(contour[:,0], contour[:,1], '.-')
plt.scatter(contour[:,0], contour[:,1], c=u)
plt.show()

In [ ]:
curvature = spline_curvature(contour, smooth=10.0)

plt.figure(figsize=(12, 8))
plt.imshow(img)
plt.scatter(contour[:,0], contour[:,1], c=curvature)

r = [50,230, 0,170]
plt.xlim(r[0],r[1]); plt.ylim(r[3],r[2])
plt.clim(-0.5, 0.5)

plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
anchor = contour[0]
eigvals, eigvecs = main_axes_from_anchor(contour, anchor)

id_right, id_left = extreme_points_perpendicular(contour, anchor, eigvecs)

intervals = low_curvature_intervals_include_points(curvature, threshold=0.02, indices_to_include=[id_right, id_left])

trimmed_intervals = trim_low_curvature_intervals(curvature, intervals, negative_threshold=-0.005)

labels = generate_labels_closed_contour(len(contour), 0, trimmed_intervals[0], trimmed_intervals[1])

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.scatter(contour[:,0], contour[:,1], c=u)
plt.plot(contour[id_right,0], contour[id_right,1], 'r*')
plt.plot(contour[id_left,0], contour[id_left,1], 'g*')

ids_right = np.arange(intervals[0][0], intervals[0][1])
ids_left = np.arange(intervals[1][0], intervals[1][1])
plt.plot(contour[ids_right,0], contour[ids_right,1], 'r.')
plt.plot(contour[ids_left,0], contour[ids_left,1], 'g.')

ids_right_trim = np.arange(trimmed_intervals[0][0], trimmed_intervals[0][1])
ids_left_trim = np.arange(trimmed_intervals[1][0], trimmed_intervals[1][1])
plt.plot(contour[ids_right_trim,0], contour[ids_right_trim,1], 'b.')
plt.plot(contour[ids_left_trim,0], contour[ids_left_trim,1], 'b.')

U = eigvecs[:,0]
V = eigvecs[:,1]

sz = 200
plt.plot([anchor[0],anchor[0]+ sz*U[0]], [anchor[1],anchor[1]+ sz*U[1]], 'r-')
plt.plot([anchor[0]-sz*V[0]/2,anchor[0]+sz*V[0]/2], [anchor[1]-sz*V[1]/2,anchor[1]+sz*V[1]/2], 'g-')

r = [50,230, 0,170]
plt.xlim(r[0],r[1]); plt.ylim(r[3],r[2])
plt.clim(-0.5, 0.5)

plt.tight_layout()
plt.show()


print(trimmed_intervals)

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.scatter(contour[:,0], contour[:,1], c=labels)
plt.plot(contour[0,0], contour[0,1], 'r*')  # stem anchor
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_xy_curvature(u, x, y, curvature):

    fig, axes = plt.subplots(
        3, 1,
        sharex=True,
        figsize=(8, 6)
    )

    # x(u)
    axes[0].plot(u, x)
    axes[0].set_ylabel("x(u)")
    axes[0].grid(True)

    # y(u)
    axes[1].plot(u, y)
    axes[1].set_ylabel("y(u)")
    axes[1].grid(True)

    # curvature(u)
    axes[2].plot(u, curvature)
    axes[2].set_ylabel("curvature")
    axes[2].set_xlabel("u (arc-length parameter)")
    axes[2].grid(True)

    plt.tight_layout()
    plt.show()

u = range(len(contour))
plot_xy_curvature( u, contour[:,0], contour[:,1], curvature )

# Development and debugging details

In [ ]:
import rembg

In [ ]:
from rembg import remove
from skimage.morphology import binary_erosion, disk


# Apply rembg to remove the background
output_img_rgba = remove(img)

# Extract the alpha channel which serves as the mask
alpha_channel = output_img_rgba[:, :, 3]

# Create a binary mask (0 or 255) from the alpha channel
binary_mask = (alpha_channel > 128).astype(bool)



# Display the binary mask
plt.figure(figsize=(8, 4))
plt.imshow(alpha_channel, cmap='gray')
plt.title('Alpha mask of Segmented Object')
plt.axis('off')
plt.colorbar()
plt.show()

plt.figure()
plt.imshow(img)
plt.contour(binary_mask, levels=[0.5])
plt.show()

In [ ]:
from skimage.measure import label, regionprops

# Label connected components in the binary mask
labeled_mask = label(binary_mask)

# Get region properties
regions = regionprops(labeled_mask)

# Find the largest region by area
largest_region = None
if regions:
    largest_region = max(regions, key=lambda r: r.area)

# Create a new mask containing only the largest region directly from its label
largest_region_mask = np.zeros_like(binary_mask, dtype=np.uint8)
if largest_region is not None:
    largest_region_mask = (labeled_mask == largest_region.label).astype(np.uint8) * 255

# Display the mask of the largest region
plt.figure(figsize=(6, 6))
plt.imshow(largest_region_mask, cmap='gray')
plt.title('Mask of the Largest Connected Component')
plt.axis('off')
plt.show()

# Fourier texture analysis

In [ ]:
# from pathlib import Path
# import skimage.io as io
# from skimage.color import rgb2gray
# import numpy as np
# import matplotlib.pyplot as plt

# IMGDIR = Path('/content/')
# #filenames = ['Picture1.jpg','Picture2.jpg','Picture3.jpg','Picture4.jpg']

# id = 0
# imgpath = IMGDIR / filenames[id]
# maskpath = IMGDIR / (filenames[id]+'.mask.png')
# img = io.imread( imgpath )
# mask = io.imread( maskpath )
# mask = (mask>0)

# gray_img = rgb2gray(img)
# #mask = rgb2gray(mask)
# print("Image converted to grayscale.")

# from scipy.ndimage import distance_transform_edt

# feather_dist = 10 # low-res
# feather_dist = 50 # high res
# dist = distance_transform_edt(mask)
# soft_mask = np.clip(dist / feather_dist, 0, 1)
# soft_mask = 0.5 * (1 - np.cos(np.pi * soft_mask))

# inner_mask = soft_mask>0.9
# masked = (gray_img).astype(float)
# masked = masked - np.min(masked[inner_mask])
# masked = masked / np.max(masked[inner_mask])
# masked = (masked-0.5) * (soft_mask)

In [ ]:
# fid,axes = plt.subplots(1,2,figsize=(12,8))
# axes[0].imshow(img)
# axes[1].imshow(soft_mask)

# fig = plt.figure()
# plt.imshow(masked)
# plt.colorbar()


In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# # Perform 2D Fourier Transform
# f_transform = np.fft.fft2(masked-masked.mean())

# # Shift the zero-frequency component to the center
# f_transform_shifted = np.fft.fftshift(f_transform)

# ## Mask center frequencies

# # Calculate radial frequencies for every point
# rows, cols = masked.shape

# # Generate frequency arrays
# freq_rows = np.fft.fftfreq(rows, d=1)
# freq_cols = np.fft.fftfreq(cols, d=1)

# # Shift frequencies to match fftshift output
# freq_rows_shifted = np.fft.fftshift(freq_rows)
# freq_cols_shifted = np.fft.fftshift(freq_cols)

# # Create 2D arrays for frequencies for each dimension
# freq_rows_2d = np.tile(freq_rows_shifted.reshape(-1, 1), (1, cols))
# freq_cols_2d = np.tile(freq_cols_shifted.reshape(1, -1), (rows, 1))


# # Calculate the radial frequency at each point
# radial_frequencies = np.sqrt(freq_rows_2d**2 + freq_cols_2d**2)

# # Define a narrow band (e.g., +/- 0.01) around the target_freq
# cut_off = 1/6   # cut-off at 10 pixels period

# # Create a boolean mask for pixels that fall within this radial frequency band
# clipped_distance = np.clip(radial_frequencies / cut_off, 0, 1)
# radial_mask = 0.5 * (1 - np.cos(np.pi * clipped_distance))

# f_transform_shifted = f_transform_shifted * radial_mask

# # Calculate the magnitude spectrum
# magnitude_spectrum = np.abs(f_transform_shifted)

# # Visualize the magnitude spectrum
# plt.figure(figsize=(10, 8))
# #plt.imshow(np.log(1 + magnitude_spectrum), cmap='gray')
# plt.imshow((magnitude_spectrum), cmap='gray')
# plt.title('Magnitude Spectrum (Log Scale)')
# plt.colorbar(label='Log Magnitude')
# plt.xlabel('Frequency (x)')
# plt.ylabel('Frequency (y)')
# plt.show()

# print("2D Fourier Transform performed and magnitude spectrum visualized.")

In [ ]:
# import numpy as np

# def angular_average(P, nbins=180):
#     ny, nx = P.shape
#     cy, cx = ny//2, nx//2

#     Y, X = np.indices((ny, nx))
#     Xc = X - cx
#     Yc = Y - cy

#     theta = np.arctan2(Yc, Xc)  # range [-pi, pi]

#     # Map to [0, pi) because spectrum is symmetric
#     theta = np.mod(theta, np.pi)

#     theta_bin = (theta / np.pi * nbins).astype(int)

#     # Optional: remove DC region
#     r = np.sqrt(Xc**2 + Yc**2)
#     mask = r < (ny//2)   # isotropic mask

#     tbin = np.bincount(theta_bin[mask].ravel(),
#                        P[mask].ravel(),
#                        minlength=nbins)

#     nbin = np.bincount(theta_bin[mask].ravel(),
#                        minlength=nbins)

#     angular_prof = tbin / nbin
#     return angular_prof, theta

# angular_acc, theta = angular_average(magnitude_spectrum**2)
# plt.plot( angular_acc )
# a = np.argmax(angular_acc)
# ap = angular_acc[a]
# plt.plot(a,ap,'r+')

# ny, nx = magnitude_spectrum.shape
# cx,cy = nx//2, ny//2
# r = 200

# plt.figure()
# #plt.imshow(theta)
# plt.imshow(np.log(1 + magnitude_spectrum), cmap='gray')
# plt.plot( [cx,cx-np.cos(a/180*np.pi)*r], [cy,cy-np.sin(a/180*np.pi)*r], 'r-*' )

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt


# # Calculate the radial frequency at each point
# radial_frequencies = np.sqrt(freq_rows_2d**2 + freq_cols_2d**2)

# # Define a narrow band (e.g., +/- 0.01) around the target_freq
# #lower_bound,upper_bound = 1/6,1/2  # low res
# lower_bound,upper_bound = 1/60,1/10  # high res

# # Create a boolean mask for pixels that fall within this radial frequency band
# radial_mask = (radial_frequencies >= lower_bound) & (radial_frequencies <= upper_bound)

# # Apply this mask to the magnitude_spectrum
# masked_magnitude_spectrum = np.zeros_like(magnitude_spectrum)
# masked_magnitude_spectrum[radial_mask] = magnitude_spectrum[radial_mask]


# # Find the maximum magnitude value within this masked region and its corresponding pixel coordinates
# max_magnitude_in_band = np.max(masked_magnitude_spectrum)
# max_peak_coords = np.unravel_index(np.argmax(masked_magnitude_spectrum), masked_magnitude_spectrum.shape)
# max_peak_row, max_peak_col = max_peak_coords

# # Visualize the magnitude spectrum (log-scaled) and highlight the identified maximum peak
# plt.figure(figsize=(10, 8))
# plt.imshow(np.log(1 + magnitude_spectrum), cmap='gray')
# #plt.imshow(magnitude_spectrum**0.1, cmap='gray')
# plt.title(f'Magnitude Spectrum (Log Scale)')
# plt.colorbar(label='Log Magnitude')
# plt.xlabel('Frequency (x)')
# plt.ylabel('Frequency (y)')

# # Highlight the found maximum peak
# plt.plot(max_peak_col, max_peak_row, 'r*', markersize=15, label='Max Peak in Band')

# # Show the apriori freq filter
# plt.contour(radial_mask, levels=[0.5])

# plt.legend()
# plt.show()

# print(f"Maximum magnitude in the band [{lower_bound:.2f}, {upper_bound:.2f}] is {max_magnitude_in_band:.2f} at pixel coordinates (row={max_peak_row}, col={max_peak_col}).")

In [ ]:
# import numpy as np
# from numpy.lib.stride_tricks import sliding_window_view

# def get_radial_frequencies(f_transform, shifted=True):
#   rows, cols = f_transform.shape[-2:]

#   # Generate frequency arrays
#   freq_rows = np.fft.fftfreq(rows, d=1)
#   freq_cols = np.fft.fftfreq(cols, d=1)

#   # Shift frequencies to match fftshift output
#   freq_rows_shifted = np.fft.fftshift(freq_rows)
#   freq_cols_shifted = np.fft.fftshift(freq_cols)

#   # Create 2D arrays for frequencies for each dimension
#   if (shifted):
#     freq_rows_2d = np.tile(freq_rows_shifted.reshape(-1, 1), (1, cols))
#     freq_cols_2d = np.tile(freq_cols_shifted.reshape(1, -1), (rows, 1))
#   else:
#     freq_rows_2d = np.tile(freq_rows.reshape(-1, 1), (1, cols))
#     freq_cols_2d = np.tile(freq_cols.reshape(1, -1), (rows, 1))

#   # Calculate the radial frequency at each point
#   radial_frequencies = np.sqrt(freq_rows_2d**2 + freq_cols_2d**2)

#   return radial_frequencies

# def cut_dc(f_transform_shifted, cut_off=1/6, shifted=True, radial_compensation=True):
#   # Calculate radial frequencies for every point
#   rows, cols = f_transform_shifted.shape[-2:]

#   # Generate frequency arrays
#   freq_rows = np.fft.fftfreq(rows, d=1)
#   freq_cols = np.fft.fftfreq(cols, d=1)

#   # Shift frequencies to match fftshift output
#   freq_rows_shifted = np.fft.fftshift(freq_rows)
#   freq_cols_shifted = np.fft.fftshift(freq_cols)

#   # Create 2D arrays for frequencies for each dimension
#   if (shifted):
#     freq_rows_2d = np.tile(freq_rows_shifted.reshape(-1, 1), (1, cols))
#     freq_cols_2d = np.tile(freq_cols_shifted.reshape(1, -1), (rows, 1))
#   else:
#     freq_rows_2d = np.tile(freq_rows.reshape(-1, 1), (1, cols))
#     freq_cols_2d = np.tile(freq_cols.reshape(1, -1), (rows, 1))

#   # Calculate the radial frequency at each point
#   radial_frequencies = np.sqrt(freq_rows_2d**2 + freq_cols_2d**2)

#   # Define a narrow band (e.g., +/- 0.01) around the target_freq
#   #cut_off = 1/6   # cut-off at 10 pixels period

#   # Create a boolean mask for pixels that fall within this radial frequency band
#   clipped_distance = np.clip(radial_frequencies / cut_off, 0, 1)
#   radial_mask = 0.5 * (1 - np.cos(np.pi * clipped_distance))

#   if (radial_compensation):
#     f_transform_shifted = f_transform_shifted * radial_mask * radial_frequencies
#   else:
#     f_transform_shifted = f_transform_shifted * radial_mask

#   return f_transform_shifted

# def welch2d(image, win_size=128, fft_size=128, overlap=0.5, cut_off=1/6):
#     step = int(win_size * (1 - overlap))

#     # Extract sliding windows
#     windows = sliding_window_view(image, (win_size, win_size))
#     windows = windows[::step, ::step]   # stride on first 2 dims [sh,sw, h,w]

#     # Apply 2D Hann window
#     # w1 = np.hanning(win_size)
#     # w2 = np.hanning(win_size)
#     # window = np.outer(w1, w2)
#     # Isotropic windowing
#     cy, cx = win_size//2, win_size//2
#     Y, X = np.indices((win_size, win_size))
#     R = np.sqrt((X-cx)**2 + (Y-cy)**2) / (win_size//2)
#     #R = R / R.max()  # normalize radius 0..1

#     # Example: cosine / Hann in radius
#     window = 0.5 * (1 + np.cos(np.pi * R)) * (R<=1)  # cosine taper to zero at edges
#     #window[R > 1] = 0

#     #plt.imshow(window)

#     #print(windows.shape, window.shape)

#     windows = windows * window

#     #windows = windows - windows.mean(axis=(2,3), keepdims=True)

#     # Compute FFTs
#     F = np.fft.fft2(windows, axes=(-2, -1), s=(fft_size,fft_size))

#     F = cut_dc( F, cut_off=cut_off, shifted=False )

#     # Compute power
#     P = np.abs(F)**2

#     # Average
#     P_avg = P.mean(axis=(0,1))

#     return P_avg

In [ ]:
# from scipy.ndimage import gaussian_filter

# # Apply Gaussian filter to the magnitude spectrum
# #smoothed_magnitude_spectrum = gaussian_filter(magnitude_spectrum**2, sigma=sigma)
# #smoothed_magnitude_spectrum = np.fft.fftshift( welch2d( masked-masked.mean(), win_size = 32, fft_size=256, overlap=0.1, cut_off=1/10  )**0.5)

# # High-res
# smoothed_magnitude_spectrum = np.fft.fftshift( welch2d( masked-masked.mean(), win_size = 256, fft_size=256, overlap=0.5, cut_off=1/64  )**0.5)

# # Visualize the smoothed magnitude spectrum
# plt.figure(figsize=(10, 8))
# #plt.imshow(np.log(1 + smoothed_magnitude_spectrum), cmap='gray')
# plt.imshow( smoothed_magnitude_spectrum**2 , cmap='gray')
# plt.title(f'Smoothed Magnitude Spectrum (Power Scale)')
# plt.colorbar(label='Power')
# plt.xlabel('Frequency (x)')
# plt.ylabel('Frequency (y)')

# # Add contours at period of integer pixel value
# radial_frequencies = get_radial_frequencies(smoothed_magnitude_spectrum, shifted=True)
# #CS = plt.contour(1/radial_frequencies, levels=[2,3,4,5], colors='c') # low-res
# CS = plt.contour(1/radial_frequencies, levels=[2,4,8,16,32], colors='c') # high-res
# plt.clabel(CS, inline=True, fontsize=10, fmt=lambda val: f"{val} pix")

# plt.show()

In [ ]:
# # Visualize as period,angle plot
# import numpy as np

# def polar_power_matrix(F, nbins_r=100, nbins_theta=180, rmin=0, rmax=1):
#     """
#     Convert 2D Fourier transform into power accumulated over radius and angle.

#     Parameters
#     ----------
#     F : 2D array
#         Shifted Fourier transform (DC at center)
#     nbins_r : int
#         Number of radial bins
#     nbins_theta : int
#         Number of angular bins
#     rmin : float
#         Minimum radius to include (to exclude DC / very low freq)

#     Returns
#     -------
#     polar_power : 2D array, shape (nbins_r, nbins_theta)
#         Power accumulated for each (radius, angle)
#     r_edges : array, shape (nbins_r+1,)
#         Radial bin edges (pixels)
#     theta_edges : array, shape (nbins_theta+1,)
#         Angular bin edges (radians, 0 to pi)
#     """
#     P = np.abs(F)**2  # power spectrum

#     ny, nx = F.shape
#     cy, cx = ny//2, nx//2

#     Y, X = np.indices(F.shape)
#     Xc = X - cx
#     Yc = Y - cy

#     # radial distance and angle
#     r = np.sqrt(Xc**2 + Yc**2) / np.sqrt(nx**2 + ny**2) # Normalized freq radius
#     theta = np.arctan2(Yc, Xc)          # [-pi, pi]
#     theta = np.mod(theta, np.pi)        # [0, pi)

#     # exclude very low frequencies
#     mask = r >= rmin

#     # bin edges
#     r_edges = np.linspace(rmin, r.max(), nbins_r+1)
#     theta_edges = np.linspace(0, np.pi, nbins_theta+1)

#     # digitize
#     r_idx = np.digitize(r[mask], r_edges) - 1
#     theta_idx = np.digitize(theta[mask], theta_edges) - 1

#     # accumulate power
#     polar_power = np.zeros((nbins_r, nbins_theta), dtype=float)
#     counts = np.zeros((nbins_r, nbins_theta), dtype=float)

#     for i in range(len(r_idx)):
#         ri = r_idx[i]
#         ti = theta_idx[i]
#         if 0 <= ri < nbins_r and 0 <= ti < nbins_theta:
#             polar_power[ri, ti] += P[mask].flat[i]
#             counts[ri, ti] += 1

#     # normalize
#     polar_power /= np.maximum(counts, 1)

#     return polar_power, r_edges, theta_edges

# import numpy as np

# import numpy as np

# def polar_periodogram_by_period(F, period_max=None, rmin=1):
#     """
#     Convert 2D Fourier transform F into power accumulated over
#     spatial period (pixels per cycle) and angle.

#     Parameters
#     ----------
#     F : 2D array
#         Shifted Fourier transform (DC at center)
#     period_max : int or None
#         Maximum period (pixels) to include. If None, uses image size.
#     rmin : float
#         Minimum radius (in pixels) to exclude DC

#     Returns
#     -------
#     polar_power : 2D array, shape (n_period_bins, nbins_theta)
#         Power accumulated per period and angle
#     period_bins : array
#         Period bins (pixels per cycle)
#     theta_edges : array
#         Angle bin edges (radians, 0 to pi)
#     """
#     P = np.abs(F)**2
#     ny, nx = F.shape
#     N = nx  # assuming square or use average of nx, ny for mapping

#     cy, cx = ny//2, nx//2
#     Y, X = np.indices(F.shape)
#     Xc = X - cx
#     Yc = Y - cy

#     r = np.sqrt(Xc**2 + Yc**2)
#     theta = np.arctan2(Yc, Xc)
#     theta = np.mod(theta, np.pi)  # [0, pi)

#     mask = r >= rmin

#     # Convert frequency radius to spatial period in pixels per cycle
#     # period = N / r
#     period = np.zeros_like(r)
#     period[mask] = N / r[mask]

#     # define integer period bins
#     period_bins_max = int(np.ceil(period[mask].max())) if period_max is None else period_max
#     period_bins = np.arange(1, period_bins_max+1)  # pixels per cycle
#     n_period_bins = len(period_bins)

#     # Angle bins
#     nbins_theta = 180
#     theta_edges = np.linspace(0, np.pi, nbins_theta+1)

#     theta_idx = np.digitize(theta[mask], theta_edges) - 1

#     # Radial (period) bin index
#     period_idx = np.digitize(period[mask], period_bins, right=True) - 1

#     # Accumulate power
#     polar_power = np.zeros((n_period_bins, nbins_theta), dtype=float)
#     counts = np.zeros((n_period_bins, nbins_theta), dtype=float)

#     for i in range(len(period_idx)):
#         pi = period_idx[i]
#         ti = theta_idx[i]
#         if 0 <= pi < n_period_bins and 0 <= ti < nbins_theta:
#             polar_power[pi, ti] += P[mask].flat[i]
#             counts[pi, ti] += 1

#     polar_power /= np.maximum(counts, 1)

#     return polar_power, period_bins, theta_edges

# polar_power, period_bins, theta_edges = polar_periodogram_by_period(smoothed_magnitude_spectrum, period_max = 32)

# # convert theta edges to degrees for plotting
# theta_deg = np.rad2deg(theta_edges[:-1] + 0.5*(theta_edges[1]-theta_edges[0]))

# # radial axis: period = 1/frequency (or just pixels)
# p_centers = period_bins[:-1] + 0.5*(period_bins[1]-period_bins[0])

# plt.figure(figsize=(10,4))
# plt.imshow(polar_power.T,
#            extent=[p_centers.min(), p_centers.max(), theta_deg.min(), theta_deg.max()],
#            origin='lower',
#            aspect='auto',
#            cmap='inferno')
# plt.xlabel("Period (pixels)")
# plt.ylabel("Angle (deg)")
# plt.title("Polar power spectrum")
# plt.colorbar(label="Power")
# plt.show()

In [ ]:
# peak_sigma = 0.02

# freq_peak_row = freq_rows_2d[max_peak_row,0]
# freq_peak_col = freq_cols_2d[0,max_peak_col]



# # Create a mask for only the maximum peak found in the band
# peak_mask = np.zeros_like(magnitude_spectrum, dtype=bool)
# peak_mask = np.exp(-1/2/peak_sigma**2*(( (freq_rows_2d-freq_peak_row)**2 + (freq_cols_2d-freq_peak_col)**2 ) )) + \
#               np.exp(-1/2/peak_sigma**2*(( (freq_rows_2d+freq_peak_row)**2 + (freq_cols_2d+freq_peak_col)**2 ) ))

# print("Defined peak_mask around the maximum frequency peak.")

# print(freq_peak_row, freq_peak_col)
# print(max_peak_row, max_peak_col)

In [ ]:
# plt.imshow(peak_mask)
# plt.colorbar()

In [ ]:
# f_transform_filtered = np.copy(f_transform_shifted)
# #f_transform_filtered[~peak_mask] = 0
# f_transform_filtered = f_transform_filtered * peak_mask


# magnitude_spectrum_filtered = np.abs(f_transform_filtered)

# plt.figure(figsize=(10, 8))
# plt.imshow(np.log(1 + magnitude_spectrum_filtered), cmap='gray')
# plt.title('Filtered Magnitude Spectrum (Log Scale)')
# plt.colorbar(label='Log Magnitude')
# plt.xlabel('Frequency (x)')
# plt.ylabel('Frequency (y)')
# plt.show()

# print("Filtered Fourier Transform created and its magnitude spectrum visualized.")

In [ ]:
# f_transform_filtered_shifted_back = np.fft.ifftshift(f_transform_filtered)
# reconstructed_image = np.fft.ifft2(f_transform_filtered_shifted_back)
# reconstructed_image = np.real(reconstructed_image)

# # Visualize the real part of the reconstructed image
# plt.figure(figsize=(12, 6))

# plt.subplot(1, 2, 1)
# plt.imshow(masked, cmap='gray')
# plt.title('Original Masked Image')
# plt.colorbar()

# plt.subplot(1, 2, 2)
# plt.imshow(reconstructed_image, cmap='gray')
# plt.title('Reconstructed Image from Filtered Freqs')
# plt.colorbar()

# plt.tight_layout()
# plt.show()

# print("Inverse Fourier Transform performed and reconstructed image visualized.")

In [ ]:
# plt.figure()
# plt.imshow(gray_img)
# plt.figure()
# plt.imshow(gray_img + reconstructed_image*10)